# Multi-Omics Integration in Glioblastoma (TCGA-GBM)
## MOFA+ · lifelines · scikit-learn

**Author:** Szonja | **Data:** TCGA-GBM (simulated with published biology)

---

## Project Summary

Glioblastoma Multiforme (GBM) is defined by profound molecular heterogeneity spanning transcriptomic, epigenomic, and mutational dimensions. The landmark Verhaak et al. (2010) study identified three transcriptional subtypes — **Proneural (PN)**, **Classical (CL)**, and **Mesenchymal (MES)** — with distinct biology and survival outcomes.

This notebook demonstrates a complete multi-omics integration workflow:
1. Simulate biologically realistic GBM data across three modalities
2. Fit MOFA+ to discover latent factors driving multi-omics variation
3. Link factors to patient survival via Kaplan-Meier and Cox regression
4. Compare early fusion, late fusion, and MOFA+ for survival prediction


## 1. Setup & Data Loading


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pathlib, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

BASE   = pathlib.Path(".")  # notebook should be at project root
DATA   = BASE / "data" / "simulated"
RES    = BASE / "results"
FIGS   = RES / "figures"
TABLES = RES / "tables"

expr    = pd.read_csv(DATA / "expression_matrix.csv",  index_col=0)
meth    = pd.read_csv(DATA / "methylation_matrix.csv", index_col=0)
mut     = pd.read_csv(DATA / "mutation_matrix.csv",    index_col=0)
clin    = pd.read_csv(DATA / "clinical_data.csv",      index_col=0)
factors = pd.read_csv(TABLES / "factor_scores.csv",   index_col=0)

print(f"Expression matrix : {expr.shape}")
print(f"Methylation matrix: {meth.shape}")
print(f"Mutation matrix   : {mut.shape}")
print(f"Patients          : {len(clin)}")
print(f"Subtypes          : {clin.subtype.value_counts().to_dict()}")


## 2. Biological Background

### GBM Subtypes (Verhaak et al. 2010)

| Subtype | Key mutations | Methylation | Median OS |
|---------|--------------|-------------|----------|
| Proneural | IDH1 (82%), ATRX (72%), TP53 (68%) | G-CIMP high | ~19.6 months |
| Classical | EGFR (88%), TERT (72%), CDKN2A (52%) | Intermediate | ~13.5 months |
| Mesenchymal | NF1 (42%), PTEN (54%), TP53 (62%) | Hypomethylated | ~6.6 months |

### Why Multi-Omics?

Single-modality analyses miss cross-modal coordination. For example:
- G-CIMP methylation (epigenome) co-occurs with IDH1 mutation (genome) and PDGFRA overexpression (transcriptome)
- EGFR amplification (genome) correlates with its own expression (transcriptome) and TERT promoter methylation (epigenome)

MOFA+ captures these coordinated signals as **latent factors** that span all three modalities.


## 3. Data Overview


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

colors = {"PN": "#4E79A7", "CL": "#F28E2B", "MES": "#E15759"}

# Subtype counts
vc = clin.subtype.value_counts()
axes[0].bar(vc.index, vc.values, color=[colors[s] for s in vc.index], alpha=0.85)
axes[0].set_title("Subtype composition", fontweight="bold")
axes[0].set_ylabel("Patients")

# OS distribution
for st, c in colors.items():
    mask = clin.subtype == st
    axes[1].hist(clin.loc[mask, "os_months"], bins=15, alpha=0.6, color=c, label=st)
axes[1].set_title("Overall survival distribution", fontweight="bold")
axes[1].set_xlabel("OS (months)"); axes[1].legend()

# Mutation frequency top 10
mut_freq = mut.mean().sort_values(ascending=False).head(10)
axes[2].barh(mut_freq.index[::-1], mut_freq.values[::-1], color="#E15759", alpha=0.8)
axes[2].set_title("Top 10 mutated genes", fontweight="bold")
axes[2].set_xlabel("Mutation frequency")

plt.tight_layout(); plt.show()


## 4. MOFA+ Integration

**MOFA+** factorises multiple data matrices sharing the same samples:

$$Y_m = Z W_m^T + \epsilon_m$$

where **Z** are latent factor scores (shared across all modalities) and **W_m** are the per-modality feature weights (loadings). An **ARD prior** automatically prunes irrelevant factors during training.

The model was fitted using `mofapy2` and stored as HDF5 for reproducibility.


### 4.1 Variance Explained per Factor and Modality


In [ ]:
var_df = pd.read_csv(TABLES / "variance_explained.csv", index_col=0)
var_total = pd.read_csv(TABLES / "variance_explained_total.csv", index_col=0)

print("Variance explained per factor (%):\n")
print(var_df.round(1).to_string())
print("\nTotal R^2 per modality (%):")
print(var_total.round(1).to_string())


In [ ]:
img = mpimg.imread(str(FIGS / "02_variance_explained.png"))
fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(img); ax.axis("off")
ax.set_title("Figure 2: MOFA+ Variance Explained", fontsize=12)
plt.tight_layout(); plt.show()


### 4.2 UMAP of Factor Scores

The 5-dimensional factor space is projected to 2D using UMAP for visualisation. Clear subtype separation confirms that the MOFA+ factors capture genuine biological structure.


In [ ]:
img = mpimg.imread(str(FIGS / "03_factor_umap.png"))
fig, ax = plt.subplots(figsize=(9, 6))
ax.imshow(img); ax.axis("off")
ax.set_title("Figure 3: UMAP of MOFA+ factor scores coloured by GBM subtype", fontsize=12)
plt.tight_layout(); plt.show()


### 4.3 Top Feature Weights

The highest-weight genes and CpG sites per factor reveal the biological drivers of each latent component.


In [ ]:
img = mpimg.imread(str(FIGS / "05_top_weights.png"))
fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(img); ax.axis("off")
ax.set_title("Figure 5: Top feature weights per MOFA+ factor", fontsize=12)
plt.tight_layout(); plt.show()


## 5. Survival Analysis

### 5.1 Kaplan-Meier Curves

We stratify by GBM subtype and factor terciles (low / mid / high), testing separation with the log-rank test.


In [ ]:
img = mpimg.imread(str(FIGS / "06_km_curves.png"))
fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(img); ax.axis("off")
ax.set_title("Figure 6: Kaplan-Meier survival curves", fontsize=12)
plt.tight_layout(); plt.show()


### 5.2 Cox Proportional Hazards Regression

Both univariate and multivariate Cox models were fitted using `lifelines.CoxPHFitter`. The forest plot shows hazard ratios (HR) with 95% CIs.

- HR < 1 → protective (higher factor → better survival)
- HR > 1 → risk factor (higher factor → worse survival)


In [ ]:
img = mpimg.imread(str(FIGS / "07_cox_forest.png"))
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(img); ax.axis("off")
ax.set_title("Figure 7: Cox PH forest plot (univariate and multivariate)", fontsize=12)
plt.tight_layout(); plt.show()


## 6. Integration Strategy Comparison

Five strategies compared by Harrell's C-index (5-fold CV):

| Strategy | Description |
|----------|-------------|
| Expression only | PCA(20) on expression features |
| Methylation only | PCA(20) on methylation features |
| Mutations only | PCA(20) on mutation features |
| Early fusion | Concatenate all three → PCA(50) |
| **MOFA+** | 5 latent factors (learned jointly) |
| Late fusion | Per-view CPH scores → ensemble mean |


In [ ]:
fusion_df = pd.read_csv(TABLES / "fusion_comparison.csv", index_col=0)
print("C-index comparison (5-fold CV):\n")
print(fusion_df.round(3).to_string())


In [ ]:
img = mpimg.imread(str(FIGS / "08_fusion_comparison.png"))
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(img); ax.axis("off")
ax.set_title("Figure 8: Integration strategy C-index comparison", fontsize=12)
plt.tight_layout(); plt.show()


In [ ]:
img = mpimg.imread(str(FIGS / "09_modality_contribution.png"))
fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(img); ax.axis("off")
ax.set_title("Figure 9: Modality contribution per MOFA+ factor", fontsize=12)
plt.tight_layout(); plt.show()


## 7. Conclusions

1. **MOFA+ recovers biologically interpretable factors.** Factor 1 captures the Proneural/Mesenchymal axis across both expression and methylation. Factor 3 captures G-CIMP methylation independently.

2. **Factor 1 is a strong independent survival predictor** (HR = 0.63, p = 1.2×10⁻¹²). Higher scores associate with Proneural biology and improved outcomes.

3. **Methylation dominates prognostic information** (C = 0.726 single-view), reflecting the known importance of G-CIMP and MGMT status in GBM.

4. **Early fusion underperforms** (C = 0.611) due to the curse of dimensionality from naively concatenating 690 features.

5. **MOFA+ achieves competitive predictive performance** (C = 0.688) with just 5 interpretable factors — a practical, scalable, and biologically meaningful integration approach.

## References

- Verhaak RGW et al. (2010). *Cancer Cell*, 17(1), 98–110.
- Argelaguet R et al. (2020). *Genome Biology*, 21, 111.
- Davidson-Pilon C (2019). *Journal of Open Source Software*, 4(40), 1317.
